# Fleet Watchlist Analysis - fixed version

This notebook is a cleaned-up replacement for the Week 3 watchlist analysis.

Fixes included:
- no GitHub token in the data URL;
- watch countries are defined inside the notebook;
- stale threshold is consistently 65 seconds;
- input columns are validated;
- output is saved as `outputs/watchlist_flagged_fixed.csv`.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd

## 2. Constants

These values should match the README, dashboard, and evaluation writeup.

In [ ]:
WATCH_COUNTRIES = [
    "Russian Federation",
    "Belarus",
    "Syrian Arab Republic",
]

STALE_THRESHOLD_SECONDS = 65

LOCAL_INPUT_CANDIDATES = [
    Path("cleaned_opensky_data.csv"),
    Path("outputs/cleaned_opensky_data.csv"),
    Path("data/processed/cleaned_opensky_data.csv"),
    Path("opensky_raw_sample.csv"),
]

REMOTE_RAW_SAMPLE_URL = "https://raw.githubusercontent.com/rmcN7/orion-t2/main/opensky_raw_sample.csv"
OUTPUT_PATH = Path("outputs/watchlist_flagged_fixed.csv")

## 3. Load data

The notebook first tries local files. If none are available, it reads the public GitHub raw sample without any token.

In [ ]:
def load_source_data():
    for path in LOCAL_INPUT_CANDIDATES:
        if path.exists():
            print(f"Loading local data: {path}")
            return pd.read_csv(path)

    print(f"Loading remote raw sample: {REMOTE_RAW_SAMPLE_URL}")
    return pd.read_csv(REMOTE_RAW_SAMPLE_URL)


df_source = load_source_data()
print(df_source.shape)
df_source.head()

## 4. Clean if needed

If the source is the raw sample, it still contains `sensors`. The original Week 2 cleaning dropped that column because it was empty for every row.

In [ ]:
def clean_if_needed(df):
    cleaned = df.copy()
    if "sensors" in cleaned.columns:
        cleaned = cleaned.drop(columns=["sensors"])
    cleaned = cleaned.sort_values(["icao24", "pulled_at"]).reset_index(drop=True)
    return cleaned


df_tracks = clean_if_needed(df_source)
print(df_tracks.shape)
df_tracks.head()

## 5. Validate columns

This makes the notebook fail clearly if the input file is not the expected OpenSky schema.

In [ ]:
REQUIRED_COLUMNS = {
    "icao24", "callsign", "origin_country", "time_position", "last_contact",
    "longitude", "latitude", "baro_altitude", "on_ground", "velocity",
    "true_track", "vertical_rate", "geo_altitude", "squawk", "spi",
    "position_source", "pulled_at",
}

missing = REQUIRED_COLUMNS - set(df_tracks.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

## 6. Filter and analyze

The key Week 3 logic: filter to watchlist countries and mark stale positions.

In [ ]:
def filter_by_country(df, countries):
    result = df[df["origin_country"].isin(countries)].copy()
    result["position_age_seconds"] = (
        result["last_contact"] - result["time_position"]
    ).astype(int)
    result["stale_position"] = result["position_age_seconds"] > STALE_THRESHOLD_SECONDS
    result["callsign"] = (
        result["callsign"]
        .fillna("(no callsign)")
        .astype(str)
        .str.strip()
        .replace("", "(no callsign)")
    )
    return result.sort_values(["origin_country", "icao24", "pulled_at"]).reset_index(drop=True)


flagged_full = filter_by_country(df_tracks, WATCH_COUNTRIES)
print(flagged_full.shape)
flagged_full.head()

## 7. Explore result

In [ ]:
print("By country:")
print(flagged_full["origin_country"].value_counts())

print("
By stale/current:")
print(flagged_full["stale_position"].value_counts())

print("
Mean location and altitude by country:")
print(
    flagged_full.groupby("origin_country")[["latitude", "longitude", "baro_altitude"]]
    .mean()
    .round(3)
)

## 8. Export

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
flagged_full.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")

## 9. Plain-language interpretation

This output is a first-pass filter. It does not prove suspicious behavior. It answers:

- Is the aircraft from a watchlist country?
- Is the displayed position current or stale?

The dashboard can then map these rows for easier analyst inspection.